pip installs (if necessary)

In [2]:
!pip install huggingface-hub
!pip install s2sphere
!pip install pymilvus
!pip install datasets
!pip install transformers 
!pip install torch 
!pip install earthengine-api 
!pip install pillow
!pip install hf_xet

some imports

In [6]:
import ee
import torch
from datasets import load_dataset
from pymilvus import MilvusClient, DataType
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from torch.utils.data import DataLoader
import os
import s2sphere
import torch.nn as nn
import torch.nn.functional as F
import shutil
import zipfile
import pandas as pd
from tqdm import tqdm
from tqdm.auto import tqdm
from queue import Queue
from threading import Thread, Lock
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download, HfApi
import queue
from queue import Queue
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = "32"

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
processor = CLIPProcessor.from_pretrained("geolocal/StreetCLIP", use_fast=True)
vision_model = CLIPModel.from_pretrained("geolocal/StreetCLIP").to(device)

Using device: cuda


auth

In [4]:
ee.Authenticate()
ee.Initialize(project=os.getenv('GEE_PROJECT_ID'))

- Initialize DB (only need to do if not created yet)
- Creating indexes for faster retrieval

In [7]:
from pymilvus import MilvusClient, DataType

# 1. Connection & Database Management
# We connect to the standalone Milvus instance and ensure the specific 'geoguessr' db exists.
client = MilvusClient(uri="http://localhost:19530", token="root:Milvus")
if "geoguessr" not in client.list_databases():
    client.create_database("geoguessr")

# Re-connect specifically to our database
client = MilvusClient(uri="http://localhost:19530", token="root:Milvus", db_name="geoguessr")

# Drop the collection if it exists to ensure the new schema is applied correctly
if client.has_collection("world_locations"):
    client.drop_collection("world_locations")

# 2. Schema Definition
# We enable dynamic_field to catch any auxiliary metadata during the OSV-5M stream.
schema = client.create_schema(auto_id=True, enable_dynamic_field=True)

# --- Primary Key ---
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)

# --- Vector Fields (The 'Targets' for our Bridge Heads) ---
# StreetCLIP: High-D visual features (768)
schema.add_field(field_name="streetclip_vec", datatype=DataType.FLOAT_VECTOR, dim=768)
# AlphaEarth: Micro-logic environmental DNA (64)
schema.add_field(field_name="alphaearth_vec", datatype=DataType.FLOAT_VECTOR, dim=64)
# Hyperbolic Tangent: Hierarchical Admin-tree logic (128) - Needed for HierLoc
schema.add_field(field_name="hyp_tangent_vec", datatype=DataType.FLOAT_VECTOR, dim=128)

# --- Scalar Metadata Fields (For Hierarchical Filtering) ---
# VARCHAR max_lengths are optimized for performance
schema.add_field(field_name="country_code", datatype=DataType.VARCHAR, max_length=10)
schema.add_field(field_name="continent", datatype=DataType.VARCHAR, max_length=50)
schema.add_field(field_name="s2_token_l10", datatype=DataType.VARCHAR, max_length=20)
schema.add_field(field_name="s2sphere_boundary", datatype=DataType.VARCHAR, max_length=512)
schema.add_field(field_name="gps", datatype=DataType.JSON)

# 3. Indexing Strategy
# We use HNSW for vectors for fast ANN search and Inverted Indexes for scalar filters.
index_params = client.prepare_index_params()

# Optimization for StreetCLIP (768D)
index_params.add_index(
    field_name="streetclip_vec",
    metric_type="COSINE",
    index_type="HNSW",
    params={"M": 16, "efConstruction": 200}
)

# Optimization for AlphaEarth (64D)
index_params.add_index(
    field_name="alphaearth_vec",
    metric_type="COSINE",
    index_type="HNSW",
    params={"M": 16, "efConstruction": 200}
)

# Scalar Indexes (Essential for the "Scout" filtering logic)
# This allows 'country == US' queries to be near-instant
index_params.add_index(field_name="country_code", index_type="INVERTED")
index_params.add_index(field_name="s2_token_l10", index_type="INVERTED")

# 4. Final Creation
client.create_collection(
    collection_name="world_locations",
    schema=schema,
    index_params=index_params
)

print("Milvus 'world_locations' collection initialized with Hierarchical Schema.")

2026-02-16 23:47:42,290 [ERROR][handler]: RPC error: [load_collection], <MilvusException: (code=65535, message=there is no vector index on field: [hyp_tangent_vec], please create index firstly)>, <Time:{'RPC start': '2026-02-16 23:47:42.287991', 'RPC error': '2026-02-16 23:47:42.289968'}>
Traceback:
Traceback (most recent call last):
  File "c:\Users\mmorr\anaconda3\Lib\site-packages\pymilvus\decorators.py", line 387, in handler
    return func(*args, **kwargs)
  File "c:\Users\mmorr\anaconda3\Lib\site-packages\pymilvus\decorators.py", line 446, in handler
    return func(self, *args, **kwargs)
  File "c:\Users\mmorr\anaconda3\Lib\site-packages\pymilvus\decorators.py", line 320, in handler
    raise e from e
  File "c:\Users\mmorr\anaconda3\Lib\site-packages\pymilvus\decorators.py", line 290, in handler
    return func(*args, **kwargs)
  File "c:\Users\mmorr\anaconda3\Lib\site-packages\pymilvus\client\grpc_handler.py", line 1639, in load_collection
    check_status(response)
    ~~~~~~

MilvusException: <MilvusException: (code=65535, message=there is no vector index on field: [hyp_tangent_vec], please create index firstly)>

Create Milvus Collection

In [ ]:
if client.has_collection(collection_name="world_locations"):
    print("Collection exists in 'geoguessr' db -> Replacing")
    client.drop_collection(collection_name="world_locations")

client.create_collection(
    collection_name="world_locations",
    schema=schema,
    index_params=index_params
)

Collection exists in 'geoguessr' db -> Replacing


Feature Processing

In [ ]:
def get_s2_token(lat, lon, level=12):
    p = s2sphere.LatLng.from_degrees(lat, lon)
    return s2sphere.CellId.from_lat_lng(p).parent(level).to_token()

def get_streetclip_embedding(images):
    inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        features = vision_model.get_image_features(**inputs)
    return features.cpu().numpy()

def alpha_earth_features(coords_list):
    points = [ee.Feature(ee.Geometry.Point([lon, lat])) for lat, lon in coords_list]
    fc = ee.FeatureCollection(points)
    ae_img = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").first()
    sampled = ae_img.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=10)
    results = []
    for feat in sampled.getInfo()['features']:
        props = feat['properties']
        vec = [float(props.get(f'A{i:02d}', 0.0) or 0.0) for i in range(64)]
        results.append(vec)
    return results

In [ ]:
!pip install huggingface_hub[hf_xet]

Batches, then deletes after each sequential zip file

In [ ]:
BATCH_SIZE = 32           # Safe for consumer GPUs (RTX 2080/3080/4080)
DB_FLUSH_SIZE = 320       # Records to buffer before inserting to Milvus
MAX_BUFFERED_ZIPS = 2     # Local disk buffer limit
MAX_INSERT_RETRIES = 3
HF_TOKEN = os.getenv("HF_TOKEN")
GEE_PROJECT = os.getenv("GEE_PROJECT_ID")
CHECKPOINT_FILE = "pipeline_checkpoint.json"

# --- Full Continent Mapping ---
CC_TO_CONTINENT = {}
for cc, cont in [
    *[(c, 'Africa') for c in 'DZ AO BJ BW BF BI CM CV CF TD KM CG CD CI DJ EG GQ ER SZ ET GA GM GH GN GW KE LS LR LY MG MW ML MR MU MA MZ NA NE NG RW ST SN SC SL SO ZA SS SD TZ TG TN UG ZM ZW RE YT'.split()],
    *[(c, 'Asia') for c in 'AF AM AZ BH BD BT BN KH CN CY GE IN ID IR IQ IL JP JO KZ KW KG LA LB MY MV MN MM NP KP OM PK PS PH QA SA SG KR LK SY TW TJ TH TL TR TM AE UZ VN YE HK MO'.split()],
    *[(c, 'Europe') for c in 'AL AD AT BY BE BA BG HR CZ DK EE FI FR DE GR HU IS IE IT XK LV LI LT LU MT MD MC ME NL MK NO PL PT RO RU SM RS SK SI ES SE CH UA GB VA FO GI JE GG IM AX'.split()],
    *[(c, 'North America') for c in 'AG BS BB BZ CA CR CU DM DO SV GD GT HT HN JM MX NI PA KN LC VC TT US PR VI GL BM AW CW SX MQ GP TC KY MS AI'.split()],
    *[(c, 'South America') for c in 'AR BO BR CL CO EC GY PY PE SR UY VE GF FK'.split()],
    *[(c, 'Oceania') for c in 'AU FJ KI MH FM NR NZ PW PG WS SB TO TV VU NC PF GU AS CK NU TK WF MP'.split()],
    *[(c, 'Antarctica') for c in ['AQ']],
]: CC_TO_CONTINENT[cc] = cont

# --- Initialization ---
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = CLIPProcessor.from_pretrained("geolocal/StreetCLIP", use_fast=True)
vision_model = CLIPModel.from_pretrained("geolocal/StreetCLIP").to(device).half().eval()

ee.Initialize(project=GEE_PROJECT)
client = MilvusClient(uri="http://localhost:19530", token="root:Milvus", db_name="geoguessr")

download_queue = queue.Queue(maxsize=MAX_BUFFERED_ZIPS)
upload_queue = queue.Queue(maxsize=10)

class CheckpointManager:
    def __init__(self, filepath=CHECKPOINT_FILE):
        self.filepath = filepath
        self.processed_zips = self._load()
    def _load(self):
        if os.path.exists(self.filepath):
            with open(self.filepath, "r") as f: return set(json.load(f))
        return set()
    def save(self, zip_name):
        self.processed_zips.add(zip_name)
        with open(self.filepath, "w") as f: json.dump(list(self.processed_zips), f)
    def is_processed(self, zip_name):
        return zip_name in self.processed_zips

def get_country_info(coords_list):
    """Uses reverse_geocoder to get ISO country codes from GPS."""
    results = rg.search(coords_list)
    return [{"country_code": r['cc'], "continent": CC_TO_CONTINENT.get(r['cc'], 'Unknown')} for r in results]

def milvus_worker():
    """Background thread to handle Milvus database insertions."""
    while True:
        records = upload_queue.get()
        if records is None: break
        for attempt in range(1, MAX_INSERT_RETRIES + 1):
            try:
                client.insert(collection_name="world_locations", data=records)
                break
            except Exception as e:
                if attempt == MAX_INSERT_RETRIES: print(f" [!!] Milvus Drop: {e}")
                else: time.sleep(2 ** attempt)
        upload_queue.task_done()

def download_worker(zip_names):
    """Background thread to stream OSV-5M zip files from Hugging Face."""
    for name in zip_names:
        try:
            path = hf_hub_download(repo_id="osv5m/osv5m", filename=f"images/train/{name}",
                                   repo_type="dataset", token=HF_TOKEN)
            download_queue.put(path)
        except Exception as e: print(f" [!] HF Download Error: {e}")
    download_queue.put(None)

@torch.inference_mode()
def process_batch(batch_images, batch_coords, pool):
    """Main processing logic: Parallelizes GEE, Reverse Geocoding, and CLIP."""
    # 1. Physical DNA (Network Bound - Google Earth Engine)
    ae_future = pool.submit(alpha_earth_features, batch_coords)
    # 2. Administrative Hierarchy (CPU Bound - Reverse Geocoder)
    geo_future = pool.submit(get_country_info, batch_coords)
    
    # 3. Visual Features (GPU Bound - StreetCLIP)
    clip_vecs = get_streetclip_embedding(batch_images, processor, vision_model, device)
    
    # Synchronize parallel tasks
    ae_vecs = ae_future.result()
    geo_info = geo_future.result()

    records = []
    for i in range(len(batch_images)):
        lat, lon = batch_coords[i]
        records.append({
            "streetclip_vec": clip_vecs[i].tolist(),
            "alphaearth_vec": ae_vecs[i],
            "gps": {"lat": float(lat), "lon": float(lon)},
            "country_code": geo_info[i]['country_code'],
            "continent": geo_info[i]['continent'],
            "s2_token_l10": get_s2_token(lat, lon, 10),
            "s2sphere_boundary": get_s2_token(lat, lon, 12),
            "hyp_tangent_vec": [0.0] * 128 # Placeholder to be updated during training
        })
    return records

def run_pipeline():
    checkpoint = CheckpointManager()
    
    # Low_memory=False resolves the DtypeWarning for large CSVs
    print(">>> Downloading Metadata...")
    csv_path = hf_hub_download(repo_id="osv5m/osv5m", filename="train.csv", repo_type="dataset", token=HF_TOKEN)
    df = pd.read_csv(csv_path, low_memory=False)
    coords_lookup = df.set_index("id")[["latitude", "longitude"]].to_dict("index")
    valid_ids = set(coords_lookup.keys())

    # Targeted 20% data split (00.zip to 19.zip)
    all_zips = [f"{str(i).zfill(2)}.zip" for i in range(20)]
    zips_to_process = [z for z in all_zips if not checkpoint.is_processed(z)]

    # Start background threads
    Thread(target=download_worker, args=(zips_to_process,), daemon=True).start()
    Thread(target=milvus_worker, daemon=True).start()

    db_buffer = []
    pool = ThreadPoolExecutor(max_workers=4)

    try:
        while True:
            zip_path = download_queue.get()
            if zip_path is None: break
            
            z_name = os.path.basename(zip_path)
            print(f"\n>>> Processing {z_name}")
            
            with zipfile.ZipFile(zip_path, "r") as zf:
                jpg_names = [f for f in zf.namelist() if f.lower().endswith(".jpg")]
                batch_imgs, batch_coords = [], []

                for name in tqdm(jpg_names, desc=f"Inference {z_name}", leave=False):
                    img_id = os.path.basename(name).split(".")[0]
                    if img_id not in valid_ids: continue
                    
                    try:
                        with zf.open(name) as f:
                            batch_imgs.append(Image.open(f).convert("RGB"))
                            row = coords_lookup[img_id]
                            batch_coords.append((row["latitude"], row["longitude"]))

                        # Execute Batch
                        if len(batch_imgs) >= BATCH_SIZE:
                            db_buffer.extend(process_batch(batch_imgs, batch_coords, pool))
                            batch_imgs, batch_coords = [], []

                        # Flush to Milvus
                        if len(db_buffer) >= DB_FLUSH_SIZE:
                            upload_queue.put(db_buffer)
                            db_buffer = []
                    except Exception:
                        continue

            # Cleanup and Checkpoint
            if os.path.exists(zip_path): 
                os.remove(zip_path)
            checkpoint.save(z_name)
            download_queue.task_done()

        # Final Flush
        if batch_imgs: db_buffer.extend(process_batch(batch_imgs, batch_coords, pool))
        if db_buffer: upload_queue.put(db_buffer)
        upload_queue.put(None)
        upload_queue.join()

    finally:
        pool.shutdown()
        print("\n[SUCCESS] Ingestion Pipeline Complete.")

if __name__ == "__main__":
    run_pipeline()

NameError: name 'os' is not defined

Testing with test dataset, applying metadata to vector db

In [ ]:
'''
metadata = pd.read_csv("./datasets/osv5m/test.csv")
image_dir = "./datasets/osv5m/images/test/00"

BATCH_SIZE = 32

def load_image(row):
    img_path = os.path.join(image_dir, f"{row['id']}.jpg")
    if os.path.exists(img_path):
        return Image.open(img_path).convert('RGB'), row['latitude'], row['longitude']
    return None

db = []
batch_images = []
batch_coords = []

for idx, row in tqdm(metadata.iterrows(), total=len(metadata)):
    result = load_image(row)
    if result is None:
        continue
    
    img, lat, lon = result
    batch_images.append(img)
    batch_coords.append((lat, lon))
    
    if len(batch_images) >= BATCH_SIZE:
        clip_vecs = get_streetclip_embedding(batch_images)
        alpha_vecs = alpha_earth_features(batch_coords)
        
        for i in range(len(batch_images)):
            lat, lon = batch_coords[i]
            db.append({
                "streetclip_vec": clip_vecs[i].tolist(),
                "alphaearth_vec": alpha_vecs[i],
                "gps": {"lat": float(lat), "lon": float(lon)},
                "s2sphere_boundary": get_s2_token(lat, lon)
            })
        
        if len(db) >= 160:
            client.insert(collection_name="world_locations", data=db)
            db = []
            print(f"Inserted at {idx}")
        
        batch_images = []
        batch_coords = []
if batch_images:
    clip_vecs = get_streetclip_embedding(batch_images)
    alpha_vecs = alpha_earth_features(batch_coords)
    for i in range(len(batch_images)):
        lat, lon = batch_coords[i]
        db.append({
            "streetclip_vec": clip_vecs[i].tolist(),
            "alphaearth_vec": alpha_vecs[i],
            "gps": {"lat": float(lat), "lon": float(lon)},
            "s2sphere_boundary": get_s2_token(lat, lon)
        })

if db:
    client.insert(collection_name="world_locations", data=db)'''

  0%|          | 644/210122 [00:37<5:39:52, 10.27it/s]

Inserted at 621


  1%|          | 1340/210122 [01:15<3:58:17, 14.60it/s]

Inserted at 1313


  1%|          | 1927/210122 [01:55<16:12:46,  3.57it/s]

Inserted at 1924


... (615 outputs left out)

Inserted at 209100


100%|█████████▉| 209806/210122 [8:00:36<02:12,  2.38it/s]

Inserted at 209792


100%|██████████| 210122/210122 [8:01:31<00:00,  7.27it/s]


In [ ]:
client.flush(collection_name="world_locations")

Dataset added!